# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

# 🌐 1. Environment Setup


In [ ]:
# ── Cell 1: 💢 Environment Sync & Cache Exorcism ────────────
!pip install -q -U bitsandbytes triton transformers datasets
import os, shutil, sys, gc, torch

# 1. Nuclear Sync: Ensure fresh library code
REPO_NAME = 'EFV-nn'
if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
!git clone -q https://github.com/ey3lock3r/EFV-nn.git
if f'/kaggle/working/{REPO_NAME}/src' not in sys.path: sys.path.insert(0, f'/kaggle/working/{REPO_NAME}/src')

# 2. Path Configuration
SAVE_PATH = '/kaggle/working/ppc_3b_pretrain.pt'

# 3. Corruption Purge & Cache Wipe
if os.path.exists(SAVE_PATH) and os.path.getsize(SAVE_PATH) < 5 * 1024**3:
    print(f'🗑️ Purging corrupted remnant ({os.path.getsize(SAVE_PATH)/1e9:.2f} GB)')
    os.remove(SAVE_PATH)
if os.path.exists(SAVE_PATH + '.tmp'): os.remove(SAVE_PATH + '.tmp')

for p in ['/tmp/torchinductor_root', os.path.expanduser('~/.cache/torch')]:
    if os.path.exists(p): shutil.rmtree(p)

# 4. Checkpoint Discovery

gc.collect(); torch.cuda.empty_cache()
# 4. Checkpoint Discovery (Main > Prev > Fresh)
if os.path.exists(SAVE_PATH) and os.path.getsize(SAVE_PATH) > 5 * 1024**3:
    LOAD_PATH = SAVE_PATH
else:
    candidates = [f for f in os.listdir('.') if f.endswith('.pt') and '_prev' in f]
    healthy = [f for f in candidates if os.path.getsize(f) > 5 * 1024**3]
    LOAD_PATH = max(healthy, key=os.path.getsize) if healthy else SAVE_PATH

print(f'✅ Environment Ready. Load Target: {LOAD_PATH}')


# 🔑 2. Secrets & Monitoring


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
import os, wandb
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    WANDB_KEY = secrets.get_secret('WANDB_API_KEY')
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print('✅ Logged into W&B and HF')
except Exception as e:
    print(f'⚠️ Secrets failure: {e}. Training will continue without W&B.')


# 📦 3. Data Pipeline


In [ ]:
# ── Cell 3: Data Pipeline (FineWeb-Edu Streaming) ────────────
from datasets import load_dataset
import torch

def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    def gen():
        buffer = []
        for ex in ds:
            buffer.extend(tokenizer(ex['text'])['input_ids'])
            while len(buffer) >= (batch_size * seq_len):
                yield torch.tensor(buffer[:batch_size * seq_len]).view(batch_size, seq_len)
                buffer = buffer[batch_size * seq_len:]
    return gen()


# 📡 4. W&B Rescue (Optional)


In [ ]:
# ── Cell 4: Optional 🛟 W&B Rescue (Pull checkpoint from cloud) ─────────────────
import shutil, os, wandb
RESCUE_RUN_PATH = "dhinsresearch/ppc-v3-automated/rn11y61i" # Updated from User URL
RESCUE_FILE = "ppc_3b_pretrain.pt"

def rescue_checkpoint(run_path, filename, target_path):
    print(f"📡 Syncing {filename} from {run_path}...")
    try:
        # 1. Download file
        restored_file = wandb.restore(filename, run_path=run_path)
        if restored_file is None: raise FileNotFoundError(f"{filename} not found in {run_path}")
        
        source_path = os.path.abspath(restored_file.name)
        target_path = os.path.abspath(target_path)

        # 2. Logic Check: Are we already there?
        if source_path == target_path:
            print(f"✅ File already at target destination: {target_path}")
            return

        # 3. Safe Move
        if os.path.exists(target_path):
            print(f'🗑️ Removing existing stale file: {target_path}')
            os.remove(target_path)

        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        shutil.move(source_path, target_path)
        print(f"🚚 Moved from {source_path} → {target_path}")
        print(f"✅ Rescue Successful!")
        
    except Exception as e:
        print(f"❌ Rescue Failed: {e}")

# Execute Rescue
rescue_checkpoint(RESCUE_RUN_PATH, RESCUE_FILE, SAVE_PATH)


# 🏗️ 5. Architectural Setup


In [ ]:
# ── Cell 5: 🏗️ Architectural Setup & v3 State Management ──
import torch, gc, importlib, os
from efv_nn.ppc_sharded import ShardedPPCGraphLLM
from transformers import AutoTokenizer

# 1. Config
VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
PRIME_DELAYS = [1, 2, 3, 5, 7, 11, 13, 17]
RESUME = False # Set to False for the very first v3 run
LOAD_PATH = '/kaggle/working/ppc_3b_pretrain.pt'

# 🚨 Toggle Diagnostics (Set "1" to enable NaN sniffing)
os.environ["PPC_DEBUG"] = "1"


# 2. Model
print('🏗️ Instantiating 3.2B PPC-OCNS v3...')
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', token=os.environ.get('HF_TOKEN'))
model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, prime_delays=PRIME_DELAYS)

# 3. Atomic State Resume
START_STEP, START_PHASE = 0, 0
OPT_STATE = None
if RESUME and os.path.exists(LOAD_PATH):
    print(f'🔄 Resuming from {LOAD_PATH}...')
    ckpt = torch.load(LOAD_PATH, map_location='cpu', mmap=True, weights_only=False)
    START_STEP = ckpt.get('step', 0)
    START_PHASE = ckpt.get('phase', 0)
    model.load_state_dict(ckpt['model_state'], strict=False)
    OPT_STATE = ckpt.get('opt_state')
    del ckpt; gc.collect()

print(f'✅ Ready. Step: {START_STEP} | Phase: {START_PHASE} | Params: {model.total_params/1e9:.2f}B')


# 🔬 6. Model Identity Audit


In [ ]:
# ── Cell 6: 🔬 Model Identity Audit (Sanity Check) ──
def run_identity_audit(model, path):
    def _audit_weights(label, weights):
        w_mean = weights.abs().mean().item()
        w_std  = weights.std().item()
        status = "✅ LEARNED" if w_std > 0.05 else "❌ RANDOM/NOISE"
        print(f"[{label}] Mean: {w_mean:.6f} | Std: {w_std:.6f} | Status: {status}")
        return w_std > 0.05

    print("🔍 Auditing Memory vs Disk...")
    # 1. Memory
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    memory_learned = _audit_weights("MEMORY", raw_model.embedding.weight)
    
    # 2. Disk
    if os.path.exists(path):
        try:
            ckpt = torch.load(path, map_location='cpu', mmap=True)
            s_dict = ckpt['model_state'] if 'model_state' in ckpt else ckpt
            w_disk = s_dict.get('embedding.weight') or s_dict.get('_orig_mod.embedding.weight')
            if w_disk is not None:
                _audit_weights("DISK  ", w_disk.float())
            else:
                print("⚠️ DISK: Could not find 'embedding.weight' in checkpoint.")
        except Exception as e:
            print(f"⚠️ DISK: Failed to read checkpoint: {e}")
    else:
        print("⚪ DISK: No checkpoint found at target path.")
    
    if not memory_learned:
        print("\n❗ WARNING: Model in memory is RANDOM. Please verify your load logic.")

# 🚀 To perform a sanity check, uncomment the line below:
# run_identity_audit(model, LOAD_PATH)


# 🚀 7. Production Training Loop


In [ ]:
# ── Cell 7: 🦸‍♂️ Automated Pilot & Memory Guard ──
import time, wandb, gc, math, torch, os
import bitsandbytes.optim as bnb
import torch.nn.functional as F

def save_checkpoint(model, opt, step, path, phase):
    gc.collect(); torch.cuda.empty_cache()
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    state = {k: v.cpu().half() for k, v in raw_model.state_dict().items()}
    torch.save({'step': step, 'phase': phase, 'model_state': state, 'opt_state': opt.state_dict()}, path + '.tmp')
    os.replace(path + '.tmp', path)
    wandb.save(path, base_path="/kaggle/working/", policy="now")
    print(f'✅ Step {step} (Phase {phase}) Cloud-Synced.')

def run_mini_audit(model, tokenizer, step):
    gc.collect(); torch.cuda.empty_cache()
    prompt = "The fundamental principles of biology include"
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"].to(model.device0)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=30, local_iters=16)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'\n🗣️ Audit (St {step}): {text}')
    wandb.log({"audit_sample": wandb.Html(f"<b>Step {step}:</b> {text}")}, step=step)

def train_automated(model, tokenizer, start_step=0, start_phase=0, opt_state=None):
    gc.collect(); torch.cuda.empty_cache()
    dataloader = get_dataloader(tokenizer)
    run = wandb.init(project='ppc-v3-automated', name=f'pilot-{time.strftime("%H%M")}')
    
    PHASES = {
        0: {'name': 'Init',       'lr': 2e-4,   'iters': 12, 'target_e': 100.0, 'tol': 1.0},
        1: {'name': 'Accel',      'lr': 1.5e-4, 'iters': 20, 'target_e': 10.0,  'tol': 0.1},
        2: {'name': 'DeepConv',   'lr': 1e-4,   'iters': 32, 'target_e': 1.0,   'tol': 0.01},
        3: {'name': 'Crystallize','lr': 5e-5,   'iters': 48, 'target_e': 0.1,   'tol': 0.0001}
    }
    
    current_phase = start_phase
    def get_opt(p_idx):
        lr = PHASES[p_idx]['lr']
        spec_params = [p for n, p in model.named_parameters() if 'spectral' in n]
        base_params = [p for n, p in model.named_parameters() if 'spectral' not in n]
        return bnb.PagedAdamW8bit([{'params': base_params, 'lr': lr}, {'params': spec_params, 'lr': lr*10}])

    opt = get_opt(current_phase)
    if opt_state is not None:
        print("🔄 Restoring optimizer momentum from memory...")
        try:
            opt.load_state_dict(opt_state)
            for param, state in opt.state.items():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(param.device)
        except Exception as e: print(f"⚠️ Could not restore optimizer state: {e}")
    
    rolling_e = []
    print(f'🚀 Pilot Engaged: Phase {current_phase} ({PHASES[current_phase]["name"]})')
    
    for step, batch in enumerate(dataloader):
        if step < start_step: continue
        
        # --- PPC-GNN SURGICAL MATRIX (Hyper-Drive Auto-Surgeon) ---
        if len(rolling_e) > 50:
            avg_e = sum(rolling_e[-50:]) / 50
            avg_l = sum(rolling_l[-50:]) / 50 if len(rolling_l) >= 50 else 10.0
            e_trend = (sum(rolling_e[-10:])/10) / (sum(rolling_e[-50:-40])/10) if len(rolling_e) >= 50 else 1.0
            l_delta = abs(avg_l - (sum(rolling_l[-100:-50])/50)) if len(rolling_l) >= 100 else 1.0
            
            # Scenario 1 & 6: Divergence / Jitter Spiral
            if e_trend > 1.5 or (avg_e > 5000 and step > 100):
                for pg in opt.param_groups: pg['lr'] *= 0.5
                print(f'\n⚠️ SURGERY: Divergence Spiral (E_trend: {e_trend:.2f}). Cooling LR.'); rolling_e = []
            
            # Scenario 2: The Noise Wall (Stagnant High Energy)
            elif avg_e > 3000 and step > 1000 and l_delta < 0.001:
                for pg in opt.param_groups: pg['lr'] *= 1.5
                print(f'\n⚡ SURGERY: Noise Wall detected. Phase Shock applied (LR+50%).')
            
            # Scenario 3: Syntactic Plateau (Word Salad)
            elif l_delta < 0.005 and avg_e < 200 and step > 500:
                for layer in model.layers:
                    layer.spectral_gate.spectral_blend.data += 0.05
                print(f'\n🥗 SURGERY: Syntactic Plateau. Increasing Expert Pressure (B+0.05).')
            
            # Standard Phase Promotion
            elif avg_e < PHASES[current_phase]['target_e'] and current_phase < 3:
                current_phase += 1
                print(f'\n🎊 Phase Promotion -> {current_phase} ({PHASES[current_phase]["name"]})')
                for pg_idx, pg in enumerate(opt.param_groups):
                    pg['lr'] = PHASES[current_phase]['lr'] * (10 if pg_idx == 1 else 1)
                rolling_e = []; rolling_l = []
        ids, targets = batch[:, :-1], batch[:, 1:].to(model.device1)
        opt.zero_grad()
        t_loop = time.time()
        with torch.amp.autocast('cuda'):
            # APD Relaxation: Pass the rolling energy mean to allow dynamic early exits
            current_rolling_e = sum(rolling_e[-10:]) / 10 if len(rolling_e) >= 10 else None
            logits, avg_iters, avg_energy, _ = model(ids, local_iters=PHASES[current_phase]['iters'], rolling_energy=current_rolling_e)
            loss = F.cross_entropy(logits.reshape(-1, 128256), targets.reshape(-1))
            ppl = math.exp(min(loss.item(), 20))
        
        loss.backward()
        opt.step()
        step_time = time.time() - t_loop
        rolling_e.append(avg_energy.item())
        rolling_l.append(loss.item())
        
        if step % 5 == 0:
            s_blend = model.layers[0].spectral_gate.spectral_blend.item()
            print(f'P{current_phase} | S{step} | L:{loss.item():.4f} | P:{ppl:.1f} | E:{avg_energy.item():.4f} | B:{s_blend:.8f} | I:{avg_iters:.1f} | T:{step_time:.2f}s')
            wandb.log({'loss': loss.item(), 'energy': avg_energy.item(), 'ppl': ppl, 'phase': current_phase, 's_blend': s_blend, 'avg_iters': avg_iters}, step=step)
        
        if step % 50 == 0: gc.collect(); torch.cuda.empty_cache()
        if (step + 1) % 500 == 0: run_mini_audit(model, tokenizer, step+1)
        if (step + 1) % 100 == 0: save_checkpoint(model, opt, step+1, SAVE_PATH, current_phase)
    wandb.finish()


# 📈 8. Training Phases


In [ ]:
# 🚀 Engage Automated Pilot
train_automated(model, tokenizer, start_step=START_STEP, start_phase=START_PHASE, opt_state=OPT_STATE)


# 🧠 9. Instant Verification


In [ ]:
# ── Cell 9: 🗣️ Verification (Instant) ────────────
gc.collect(); torch.cuda.empty_cache()
def verify_generation(model, tokenizer, prompt="The fundamental principles of biology include"):
    # --- Atomic Audit Override: Forcing Maximum Inference Precision ---
    for layer in model.layers:
        layer.exit_threshold.data.fill_(0.00005)
        layer.min_iters = 8

    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    print("\n🧠 Regular Generation...")
    t0 = time.time()
    with torch.no_grad():
        out_reg = model.generate(inputs, max_new_tokens=80, local_iters=64)
    print(f"[Reg] {time.time()-t0:.1f}s: {tokenizer.decode(out_reg[0], skip_special_tokens=True)}")
    
    print("\n🐝 Advanced Swarm Inference (N=8)...")
    t0 = time.time()
    # Picking the most resonant ghost state (lowest energy resonance)
    with torch.no_grad():
        out_swarm = model.generate_swarm(inputs, max_new_tokens=80, swarm_size=4, local_iters=64)
    print(f"[Swarm] {time.time()-t0:.1f}s: {tokenizer.decode(out_swarm[0], skip_special_tokens=True)}")
    print("\n🧬 Running MoE Health Audit...")
    captured_indices = []
    def hook_fn(module, input, output):
        captured_indices.append(output[1].detach().cpu())
        
    hooks = []
    try:
        for name, module in model.named_modules():
            if module.__class__.__name__ == 'ExpertChoiceMoEMatcher':
                hooks.append(module.register_forward_hook(hook_fn))
                
        with torch.no_grad(), torch.amp.autocast('cuda'):
            model(inputs, local_iters=8)
            
    finally:
        for h in hooks: h.remove()
    
    B_T = inputs.shape[1]
    total_coverage = 0
    for indices in captured_indices[:24]:
        flat_indices = indices.flatten()
        unique_tokens = torch.unique(flat_indices).shape[0]
        total_coverage += (unique_tokens / float(B_T))
            
    avg_cov = total_coverage / len(captured_indices[:24])
    print(f"🔬 Average Expert Diversity Score (Coverage): {avg_cov:.1%}")
    if avg_cov < 0.3:
        print("🚨 WARNING: High Expert Cloning detected. Experts are picking the same tokens.")
    elif avg_cov > 0.8:
        print("✅ HEALTHY: Maximum MoE utilization. Experts are highly specialized.")
    else:
        print("⚠️ MODERATE: Experts are learning, but there is noticeable overlap.")

# To run: 
# verify_generation(model, tokenizer)


# 🗣️ 10. Generation Sandbox


In [ ]:
# ── Cell 10: 🧪 Generation Sandbox (Verification) ────────────
gc.collect(); torch.cuda.empty_cache()
def run_sandbox_verification(prompt='The fundamental principles of biology include'):
    # --- Atomic Audit Override: Forcing Maximum Inference Precision ---
    for layer in model.layers:
        layer.exit_threshold.data.fill_(0.00005)
        layer.min_iters = 8

    inputs = tokenizer(prompt, return_tensors='pt')['input_ids']
    print('\n🧠 Running Native Generation...')
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=80, local_iters=64)
    print(f'Output:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}')

# run_sandbox_verification()
